---

Als Struktur habe ich mir folgendes überlegt :

1. Einstellung der API um requests schicken zu können
2. Wir geben suchkriterien ein (params), in eine Strukturierter Form
3. man speichert die Repsonse der API in dict liste 
4. Man Normalisiert die Antwort der API in Begriffe, die für alle APIs relevant sind. z.B wird hier company_name in company umgewandelt.

---
1. API aufruf = Einbauung der API URL + response
2. Suchkriterien : params = get_params(search, category, company, limit)
3. Response : raw_jobs = fetch_remotive(params)
4. Normalisierung : jobs = normalize_remotive_list(raw_jobs)

Diese Sruktur wird von den Punkten 1 bis 3 jeweils für die verschiede APIs variiren. Der Punkt 4 Normalisierung sorgt dafür das es dann eine Einheit gibt und alle Daten zusammengefügt werden können.



---

# 1.API Einrichtung

Imports

In [163]:
import json
from datetime import datetime
import requests
import pandas as pd

API 

In [162]:
remotive_URL= "https://remotive.com/api/remote-jobs"
response= requests.get(remotive_URL)
print(response.status_code)



200


# 2. Suchkriterien


#### userinputs

In [146]:
search =input("Suchbegriff eingeben (leer lassen zum Überspringen): ")
category = input("Kategorie eingeben (z. B. software-dev, leer lassen): ")  
company = input("Firmenname eingeben (leer lassen): ")      
limit = input("Limit Anzahl Ergebnisse (z. B. 10, leer lassen): ") or 5

#### get_params()

In [147]:
#Funktion um unsere Variabeln für die API verständlich zu machen
# if search = True wenn NotBLank, False wenn None oder ""
# search.strip() = entfernt die Leerzeichen am Anfang und Ende
#Die Funktion üeberprüft das die variabel nicht leer sind nachdem die Leerzeichen weggenommen wurden.
# API's mögen keine Leere Variabeln, deswegen diese Funktion

def  get_params(search: str, category : str, company :str, limit : int | None = None )-> dict:
    params = {}
    if search and search.strip():
        params["search"] = search.strip()
    if category and category.strip():
        params["category"] = category.strip()
    if company and company.strip():
        params["company_name"] = company.strip()
    if limit is not None:
        params["limit"] = int(limit)
    return params


# 3.Response

In [ ]:
def fetch_remotive(params: dict) -> list[dict]:
    #  API aufrufen und in das Normalformat umwandeln.
    # params wird dieser als dict angegeben, und wenn mehr as ein dict gibt (ein job= ein dict), wird eine liste von dicts erstellt.
    r = requests.get(remotive_URL, params=params, timeout=30)
    #Anfrage (request) mit api url, unsere userinput params (recherche Angaben), wartelimit (timeout=30)
    r.raise_for_status()
    # test der Verbindung. Wenn 200, funktionniert. Wird nicht angezeigt hier. braucht dafür ein print(r.raise_for_status()) 
    data = r.json()
    #speicher der daten in data Variabel als json Format
    return r.json().get("jobs", [])
    # braucht ein return 

# 4. Normalisierung

In [ ]:
def normalize_remotive(job: dict) -> dict:
    # Normalisierung der Suchbegriffe für ein dict. Nicht für die Dict liste !
    return {
        "id": f"remotive:{job.get('id')}",
        "source": "remotive",
        "title": job.get("title"),
        "company": job.get("company_name"),
        "location": job.get("candidate_required_location"),
        "url": job.get("url"),
        "posted_at": job.get("publication_date"),
    }

In [ ]:
# Wir müssen noch die normalize_remotive einbauen, sodass sie die ganze liste Normalisiert, und nicht nur das erste dict unserer Liste
def normalize_remotive_list(jobs: list[dict]) -> list[dict]:
    return [normalize_remotive(j) for j in jobs]

In [ ]:
# Hier bauen wir die Funktionen  in usere Strukturlogik ein

params = get_params(search, category, company, limit)
raw_jobs = fetch_remotive(params)
jobs = normalize_remotive_list(raw_jobs)

In [ ]:
# Befehl an der API mit usere Suchkriterien (params)  und lassen uns die url anzeigen zum Controlling
r = requests.get(remotive_URL, params=params, timeout=30)
r.raise_for_status()
print("Aufgerufene URL:", r.url)

Aufgerufene URL: https://remotive.com/api/remote-jobs?search=data&limit=5


In [ ]:
"""data = r.json()
for j in data.get("jobs", [])[:5]:
    print("-----")
    print("Titel:", j.get("title"))
    print("Unternehmen:", j.get("company_name"))
    print("Ort:", j.get("candidate_required_location"))
    print("Link:", j.get("url"))
"""
# Wir lassen uns die r (response) in ein DF angeben
jobs= normalize_remotive_list(raw_jobs)
pd.DataFrame(jobs)

,id,source,title,company,location,url,posted_at
0,remotive:2063223,remotive,Senior Full-stack Developer,Lemon.io,"Americas, Europe, Asia, Oceania",https://remotive.com/remote-jobs/software-dev/...,2025-09-17T13:58:51
1,remotive:2062214,remotive,Data Annotator,HelixRecruit,USA,https://remotive.com/remote-jobs/writing/data-...,2025-09-15T07:07:49
2,remotive:1591692,remotive,Senior ML Engineer,Proxify,CET +/- 3 HOURS,https://remotive.com/remote-jobs/software-dev/...,2025-09-14T16:15:53
3,remotive:1680495,remotive,Office Assistant,Coalition Technologies,Worldwide,https://remotive.com/remote-jobs/marketing/off...,2025-09-11T20:31:03
4,remotive:2058383,remotive,Senior AI Product Manager,EverAI,Europe,https://remotive.com/remote-jobs/product/senio...,2025-09-10T14:02:25
